<a href="https://colab.research.google.com/github/lovey7768/enterprise-hr-agents/blob/main/Autonomous_HR_Multi_Agent_Infrastructure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Install Dependencies

# The Production Stack Breakdown

->Orchestration & State Management (LangGraph): Replaces brittle prompt chains with a stateful Directed Acyclic Graph (DAG). Every agent reads from and writes to a single typed state container

->Data Contracts (Pydantic v2): Guarantees that agents output structured JSON objects (e.g., match score as a float, missing skills as a list), eliminating hallucinated formatting.


->Inference Acceleration (Groq / Llama 3.3 or Gemini): 100% cost-free API tier providing sub-second token generation for multi-turn agent evaluation.


->Human-in-the-Loop (HITL) Gate: Prevents the system from dispatching an employment offer or rejection without human authorization.



In [ ]:
!pip install -q langgraph langchain-groq langchain-core pydantic pypdf streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.2 MB/s eta 0:00:00


# Cell 2: Configure Free LLM & Client Initialization

We will use Groq's free API with llama-3.3-70b-versatile (or llama3-8b-8192) because its inference speed keeps agent execution loops near instantaneous

In [ ]:
import os
from langchain_groq import ChatGroq
from google.colab import userdata

# Get your free Groq API key from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm=ChatGroq(
    model="openai/gpt-oss-120b", # or "qwen/qwen3.6-27b"
    temperature=0.1
)
print("LLM Engine Initialized!")

LLM Engine Initialized!


# Cell 3: Define Enterprise Data Contracts (Pydantic Schemas)
In production, agents never talk in raw strings; they pass typed contracts:

In [ ]:
from typing import List, Optional, TypedDict
from pydantic import BaseModel , Field

# Contract1: Resume Screener Output
class ScreenReport(BaseModel):
  candidate_name: str = Field(description="Name of the candidate")
  match_score: float = Field(description="Match score from 0 to 100 based on job requirements")
  matched_skills: List[str] = Field(description="Skills present in resume that match JD")
  missing_skills: List[str]= Field(description="Crucial skills requirement by JD missing from resume")
  screening_decision: str= Field(description="PASS, BORDERLINE, or REJECT")

# Contract 2: Interview Question Output
class InterviewPlan(BaseModel):
  technical_questions: List[str]=Field(description="3 deep technical questions targeting skill gaps")
  behavioral_questions: List[str]= Field(desciption="2 scenario questions evaluating team collabration")

# Contract 3: Compliance & compensation verification
class ComplianceReport(BaseModel):
  visa_complaints: bool= Field(description="Is work authorization valid?")
  compensation_bond_fit: bool = Field(description="Does salary expectation fit the budget?")
  compliance_notes: str = Field(description ="Summary of legal or internal policy flags")

# Contract 4: Final HR Decision Document
class FinalHROutput(BaseModel):
  decision: str = Field(description="OFFER , INTERVIEW, OR REJECT")
  action_item: str = Field(description="General email draft or next memo")


/tmp/ipykernel_1430/2111323271.py:15: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  behavioral_questions: List[str]= Field(desciption="2 scenario questions evaluating team collabration")


# Cell 4: Define the Shared Graph State Container
In LangGraph, all agents read from and mutate a central state dictionary:

In [ ]:
from typing import Optional, TypedDict

class HRWorkflowState(TypedDict):
  job_description: str
  resume_text: str
  candidate_experience_years: int
  salary_expectation: int
  salary_budget_max: int
  work_authorization: str
  # State  updated by agent node
  screening_result: Optional[ScreenReport]
  interview_plan: Optional[InterviewPlan]
  compliance_result: Optional[ComplianceReport]
  final_output: Optional[FinalHROutput]

# Cell 5: Build the Specialized AI Agent Nodes
Each agent is an isolated Python function that accepts the state, executes its prompt, parses structured Pydantic data, and updates the state.  

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# -----------AGENT 1: RESUME SCREENER----------------------------

def resume_screener_agent(state: HRWorkflowState) -> dict:
  print("\n [Agent 1: Resume Screener ] Evaluatting candidate against Jd....")

  structured_llm= llm.with_structured_output(ScreenReport)
  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are an expert Technical Recruiter. Analyze the candidate 's resume strictly against the job description. Output structured JSON."),
      ("human"," JOB DESCRIPTION:\n{jd}\n\nRESUME TEXT:\n{resume}")

  ])

  chain= prompt | structured_llm
  report: ScreenReport =chain.invoke({
      "jd": state["job_description"],
      "resume": state["resume_text"]
  })

  return {"screening_result": report}

# ------- AGENT 2: TECHNICAL & BEHAVIORAL INTERVIEW GENERATOR--------

def interview_auditor_agent(state: HRWorkflowState) -> dict:
    print("[Agent 2: Interview Audito] Formulating targeted assessement questions...")

    screening = state["screening_result"]
    structured_llm =llm.with_structured_output(InterviewPlan)

    prompt= ChatPromptTemplate.from_messages([
    ("system", "You are an engineering hiring manager. Create 3 targeted questions focusing on the candidate's missing skills, and 2 behavioural questions."),
    ("human","Candidate Missing Skills:{missing_skills}\nMatched Skills: {matched_skills}\nJD: {jd}")
    ])
    chain = prompt | structured_llm
    plan: InterviewPlan = chain. invoke({
        "missing_skills": ",".join(screening.missing_skills),
        "matched_skills": ",".join(screening.matched_skills),
        "jd": state["job_description"]
    })

    return {"interview_plan": plan}

# ---------AGENT 3: COMPLIANCE & POLICY AUDITOR----------------------------

def compliance_auditor_agent(state: HRWorkflowState) -> dict:
      print("[Agent 3: Compliance Auditor] Checking labor policies, visa , and salary bands....")

      structured_llm =llm.with_structured_output(ComplianceReport)
      prompt= ChatPromptTemplate.from_messages([
      ("system", "You are an HR legal and compliance officer. Verify if work authorization is supported and if candidate salary expectation fits within budget."),
      ("human", "Salary Expectation:₹{exp}\nMax Budget: ₹{budget}\nWork Authorization:{auth}")

      ])

      chain= prompt | structured_llm
      compliance: ComplianceReport = chain.invoke({
          "exp": state["salary_expectation"],
          "budget": state["salary_budget_max"],
          "auth": state["work_authorization"]
      })

      return {"compliance_result": compliance}

# -----AGENT 4 : HR OPS EXECUTIVE COMMUNICATION--------------
def hr_ops_agent(state: HRWorkflowState) -> dict:
  print("[Agent 4: HR Ops Agent] Synthesizing reports and drafting executive documents.....")

  screening =state["screening_result"]
  compliance =state["compliance_result"]
  interview= state["interview_plan"]

  structured_llm = llm.with_structured_output(FinalHROutput)

  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are an HR Operations Director. Based on screening and compliance, determine the final action (OFFER, INTERVIEW, OR REJECT) and draft a formal cummunication email."),
      ("human", "Screening Score: {score} ({decision})\nCompliance:{comp}\nInterview Plan: {plan}")

  ])

  chain = prompt | structured_llm
  decision: FinalHROutput = chain.invoke({
      "score": screening.match_score,
      "decision": screening.screening_decision,
      "comp": compliance.compliance_notes,
      "plan": str(interview)
  })
  return {"final_output": decision}

# Cell 6: Compile the LangGraph State Machine with Checkpointing & HITL
We now wire the graph with nodes, execution edges, and an interrupt breakpoint before final dispatch to simulate Human-in-the-Loop review:

In [ ]:
from langgraph.graph import StateGraph , START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. Initialize StateGraph
builder = StateGraph(HRWorkflowState)

# 2. Add Nodes
builder.add_node("screener", resume_screener_agent)
builder.add_node("interviewer", interview_auditor_agent)
builder.add_node("compliance", compliance_auditor_agent)
builder.add_node("hr_ops", hr_ops_agent)

# 3. Add Edges (Screener -> Parallel [Interviewer, Compliance] -> HR Ops)
builder.add_edge(START, "screener")
builder.add_edge("screener", "interviewer") # Screener leads to interviewer
builder.add_edge("screener", "compliance") # Screener also leads to compliance for parallel execution

builder.add_edge("interviewer","hr_ops") # Interviewer leads to hr_ops
builder.add_edge("compliance", "hr_ops") # Compliance leads to hr_ops
builder.add_edge("hr_ops", END)

# Compliance with state Checkpointer & Interrupt Before HR Ops

memory = MemorySaver()
hr_graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["hr_ops"] # HITL : Pauses execution for manager approval before writing offers
)

print("Production Multi-Agent Graph Compiled Successfully!")

Production Multi-Agent Graph Compiled Successfully!


# Cell 7: Execute the Multi-Agent HR Pipeline
Let's simulate a candidate application entering the enterprise infrastructure:

In [ ]:
import json

sample_payload = {
    "job_description": "Senior Backend & AI Engineer. Requirements: Python, FastAPI, PostgreSQL, LangGraph, Docker, Distributed Task Queues. Budget: $160,000.",
    "resume_text": "Experienced Python Software Developer with 5 years building scalable APIs using FastAPI, PostgreSQL, Docker, and Redis. Built RAG pipelines using LangChain.",
    "candidate_experience_years": 5,
    "salary_expectation": 145000,
    "salary_budget_max": 160000,
    "work_authorization": "US Citizen (No sponsorship required)",
    "screening_result": None,
    "interview_plan": None,
    "compliance_result": None,
    "final_output": None
}

# Check if resume_pdf_text exists from the PDF upload cell and use it
# This ensures that if the user uploaded a PDF, its content is used for screening.
if 'resume_pdf_text' in globals() and resume_pdf_text:
    sample_payload["resume_text"] = resume_pdf_text
    print("Using resume text from uploaded PDF.")
else:
    print("Using default hardcoded resume text.")

# Unique candidate session thread
config = {"configurable": {"thread_id": "candidate_app_001"}}

print(" Initiating Candidate Processing...")
for event in hr_graph.stream(sample_payload, config=config):
    for node_name, result in event.items():
        print(f" Step Finished: {node_name}")

Using resume text from uploaded PDF.
 Initiating Candidate Processing...

 [Agent 1: Resume Screener ] Evaluatting candidate against Jd....
 Step Finished: screener
[Agent 2: Interview Audito] Formulating targeted assessement questions...
[Agent 3: Compliance Auditor] Checking labor policies, visa , and salary bands....
 Step Finished: compliance
 Step Finished: interviewer
 Step Finished: __interrupt__


### Upload and Process Resume PDF

To input a resume as a PDF, we'll first install a library to read PDF files, then provide code to upload your PDF, extract its text, and finally update the `sample_payload` for processing.

In [ ]:
from google.colab import files
import pypdf
import io

print("Please upload your resume PDF file:")
uploaded = files.upload()

resume_pdf_text = ""
for fn in uploaded.keys():
    pdf_file = io.BytesIO(uploaded[fn])
    reader = pypdf.PdfReader(pdf_file)
    for page in reader.pages:
        resume_pdf_text += page.extract_text()

# Update the sample_payload with the extracted resume text
sample_payload["resume_text"] = resume_pdf_text

print("Resume text extracted and updated in sample_payload.")
# Display a snippet of the extracted text to confirm
print("--- Extracted Resume Snippet ---")
print(resume_pdf_text[:500] + "...")

Please upload your resume PDF file:


Saving lovepreet singh AI ENGINEER.pdf to lovepreet singh AI ENGINEER (1).pdf
Resume text extracted and updated in sample_payload.
--- Extracted Resume Snippet ---
LOVEPREET SINGH
AI / ML ENGINEER — COMPUTER VISION & NLP
Bathinda, Punjab, India  •  +91 6280214446  •  loveymann49@gmail.com
github.com/lovey7768  •  linkedin.com/in/lovepreet-singh-a839821a3  •  kaggle.com/lovepreetsingh7
EDUCATION
Bachelor of Computer Applications (BCA) 2023 – 2026
D.A.V. College, Bathinda (Affiliated with Punjabi University, Patiala) — Result Awaited
Senior Secondary (Class XII), PSEB — Non-Medical, 77% 2022 – 2023
Matriculation (Class X), CBSE — 70% 2020 – 2021
PROFESSIONAL...


# Cell 8: Human-in-the-Loop Review & Resumption
The graph paused before hr_ops. Let's inspect the intermediate state, approve, and continue execution:

In [ ]:
# Inspect current state stored in memory checkpointer
current_state = hr_graph.get_state(config)
print("⏸ [SYSTEM PAUSED AT HITL GATE]")
print(f"Next Node to Execute: {current_state.next}")
print("\n--- Screening Summary ---")
print(json.dumps(current_state.values["screening_result"].model_dump(), indent=2))

print("\n--- Generated Interview Questions ---")
print(json.dumps(current_state.values["interview_plan"].model_dump(), indent=2))

# Human Manager Decision: Resume execution
print("\n [Hiring Manager Approval Received]: Dispatching final HR Ops action...")
for event in hr_graph.stream(None, config=config):
    for node_name, result in event.items():
        print(f" Completed: {node_name}")

# Print Final Synthesized Result
final_state = hr_graph.get_state(config)
print("\n================ FINAL HR ACTION ================")
print(f"DECISION: {final_state.values['final_output'].decision}")
print(f"DRAFTED EMAIL:\n{final_state.values['final_output'].action_item}")

⏸ [SYSTEM PAUSED AT HITL GATE]
Next Node to Execute: ('hr_ops',)

--- Screening Summary ---
{
  "candidate_name": "Lovepreet Singh",
  "match_score": 17.0,
  "matched_skills": [
    "Python"
  ],
  "missing_skills": [
    "FastAPI",
    "PostgreSQL",
    "LangGraph",
    "Docker",
    "Distributed Task Queues"
  ],
  "screening_decision": "REJECT"
}

--- Generated Interview Questions ---
{
  "technical_questions": [
    "Given your experience with Python, how would you design and implement a RESTful API using FastAPI to handle high concurrency, and what specific FastAPI features would you leverage to ensure scalability and maintainability?",
    "Explain how you would model a complex relational schema in PostgreSQL for an AI-driven application, including considerations for indexing, query optimization, and handling large volumes of time\u2011series data.",
    "Describe how you would containerize a Python microservice with Docker and orchestrate it alongside a distributed task queue (e

# Streamlit APP

In [ ]:
%%writefile app.py
import os
import io
import json
import pypdf
import streamlit as st
from typing import List, Optional, TypedDict
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ─────────────────────────────────────────────────────────────
# 1. PAGE CONFIGURATION & STYLING
# ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Enterprise HR Multi-Agent Suite",
    page_icon="💼",
    layout="wide"
)

st.title("💼 Enterprise Autonomous HR Multi-Agent Suite")
st.caption("Stateful Multi-Agent Workflow powered by LangGraph, Groq, Pydantic, and Streamlit")

# ─────────────────────────────────────────────────────────────
# 2. SIDEBAR CONFIGURATION
# ─────────────────────────────────────────────────────────────
with st.sidebar:
    st.header("🔑 Model & API Credentials")
    api_key = st.text_input("Groq API Key", type="password", help="Enter your Groq API key (console.groq.com)")

    st.divider()
    st.subheader("⚙️ System Architecture")
    st.markdown("""
    - **Agent 1:** Resume Screener & Skill Matcher
    - **Agent 2:** Targeted Interview Generator
    - **Agent 3:** Labor & Comp Compliance Checker
    - **Gate:** Human-in-the-Loop Manager Checkpoint
    - **Agent 4:** Executive HR Ops & Email Dispatcher
    """)

# ─────────────────────────────────────────────────────────────
# 3. PYDANTIC SCHEMAS & STATE DEFINITION
# ─────────────────────────────────────────────────────────────
class ScreeningReport(BaseModel):
    candidate_name: str = Field(description="Name of the candidate")
    match_score: float = Field(description="Match score from 0 to 100")
    matched_skills: List[str] = Field(description="Skills present matching the JD")
    missing_skills: List[str] = Field(description="Required skills missing")
    screening_decision: str = Field(description="PASS, BORDERLINE, or REJECT")

class InterviewPlan(BaseModel):
    technical_questions: List[str] = Field(description="Targeted technical questions")
    behavioral_questions: List[str] = Field(description="Behavioral scenario questions")

class ComplianceReport(BaseModel):
    visa_compliant: bool = Field(description="Work authorization status")
    compensation_band_fit: bool = Field(description="Salary budget fit")
    compliance_notes: str = Field(description="Summary of legal or budget notes")

class FinalHROutput(BaseModel):
    decision: str = Field(description="OFFER, INTERVIEW, or REJECT")
    action_item: str = Field(description="Drafted email communication")

class HRWorkflowState(TypedDict):
    job_description: str
    resume_text: str
    salary_expectation: int
    salary_budget_max: int
    work_authorization: str
    screening_result: Optional[dict]
    interview_plan: Optional[dict]
    compliance_result: Optional[dict]
    final_output: Optional[dict]

# ─────────────────────────────────────────────────────────────
# 4. LANGGRAPH MULTI-AGENT BUILDER
# ─────────────────────────────────────────────────────────────
def build_hr_graph(groq_api_key: str):
    llm = ChatGroq(
        model_name="openai/gpt-oss-120b", # Corrected model name
        temperature=0.1,
        groq_api_key=groq_api_key
    )

    def screener_node(state: HRWorkflowState) -> dict:
        structured_llm = llm.with_structured_output(ScreeningReport)
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert Technical Recruiter. Evaluate the candidate strictly against the Job Description. Output structured JSON."),
            ("human", "JOB DESCRIPTION:\n{jd}\n\nRESUME:\n{resume}")
        ])
        chain = prompt | structured_llm
        res = chain.invoke({"jd": state["job_description"], "resume": state["resume_text"]})
        return {"screening_result": res.model_dump()}

    def interviewer_node(state: HRWorkflowState) -> dict:
        screening = state["screening_result"]
        structured_llm = llm.with_structured_output(InterviewPlan)
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an engineering hiring manager. Formulate 3 technical questions specifically targeting missing skills, and 2 behavioral questions."),
            ("human", "Missing Skills: {missing}\nMatched Skills: {matched}\nJD: {jd}")
        ])
        chain = prompt | structured_llm
        res = chain.invoke({
            "missing": ", ".join(screening.get("missing_skills", [])),
            "matched": ", ".join(screening.get("matched_skills", [])),
            "jd": state["job_description"]
        })
        return {"interview_plan": res.model_dump()}

    def compliance_node(state: HRWorkflowState) -> dict:
        structured_llm = llm.with_structured_output(ComplianceReport)
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an HR Legal and Compliance Officer.\nEvaluate work authorization and compensation fit based on the hiring context:\n\n- If the candidate holds citizenship or permanent legal right to work in the hiring country (e.g., Indian citizen for jobs in India / Remote India roles, or US citizen for US roles), mark `visa_compliant: True`.\n- Only mark `visa_compliant: False` if the candidate explicitly requires foreign visa sponsorship that the company policy does not support.\n- Check if the salary expectation is within or equal to the maximum role budget."""),
            ("human", """Hiring Context:\n- Candidate Work Authorization / Citizenship: {auth}\n- Salary Expectation: {exp}\n- Role Maximum Budget: {budget}\n- Job Description Context: {jd}""")
        ])
        chain = prompt | structured_llm
        res = chain.invoke({
            "exp": state["salary_expectation"],
            "budget": state["salary_budget_max"],
            "auth": state["work_authorization"],
            "jd": state["job_description"]
        })
        return {"compliance_result": res.model_dump()}

    def ops_node(state: HRWorkflowState) -> dict:
        screening = state["screening_result"]
        compliance = state["compliance_result"]
        interview = state["interview_plan"]
        structured_llm = llm.with_structured_output(FinalHROutput)
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an HR Operations Director. Formulate the final hiring decision and write a polished email communication."),
            ("human", """Screening Decision: {score} ({dec})\nCompliance: {comp}\nQuestions: {plan}""")
        ])
        chain = prompt | structured_llm
        res = chain.invoke({
            "score": screening.get("match_score"),
            "dec": screening.get("screening_decision"),
            "comp": compliance.get("compliance_notes"),
            "plan": json.dumps(interview)
        })
        # --- ADDED FOR DEBUGGING --- START
        print(f"[DEBUG in app.py ops_node] Final Output Action Item: {res.action_item}")
        # --- ADDED FOR DEBUGGING --- END
        return {"final_output": res.model_dump()}

    builder = StateGraph(HRWorkflowState)
    builder.add_node("screener", screener_node)
    builder.add_node("interviewer", interviewer_node)
    builder.add_node("compliance", compliance_node)
    builder.add_node("hr_ops", ops_node)

    builder.add_edge(START, "screener")
    builder.add_edge("screener", "interviewer")
    builder.add_edge("screener", "compliance")
    builder.add_edge("interviewer", "hr_ops")
    builder.add_edge("compliance", "hr_ops")
    builder.add_edge("hr_ops", END)

    if "graph_memory" not in st.session_state:
        st.session_state.graph_memory = MemorySaver()

    return builder.compile(
        checkpointer=st.session_state.graph_memory,
        interrupt_before=["hr_ops"]
    )

# ─────────────────────────────────────────────────────────────
# 5. UI CONTROLS & PDF INGESTION PIPELINE
# ─────────────────────────────────────────────────────────────
col_left, col_right = st.columns([1, 1], gap="medium")

with col_left:
    st.subheader("📋 1. Job Description & Candidate Info")
    jd_input = st.text_area(
        "Job Description (JD)",
        value="Senior AI & Backend Engineer. Requirements: Python, FastAPI, PostgreSQL, LangGraph, Docker, Distributed Task Queues. Budget: ₹16,00,000.", # Indian budget example
        height=140
    )

    col_c1, col_c2 = st.columns(2)
    with col_c1:
        salary_exp = st.number_input("Candidate Salary Expectation (₹", value=1300000, step=50000) # Indian currency
        budget_max = st.number_input("Max Role Budget (₹", value=1600000, step=50000) # Indian currency
    with col_c2:
        work_auth = st.selectbox(
            "Work Authorization / Status",
            [
                "Indian Citizen (Eligible to work in India / Remote)",
                "Authorized to Work Locally (No Sponsorship Required)",
                "Requires Visa Sponsorship / Transfer",
                "US Citizen / Green Card Holder"
            ]
        )

with col_right:
    st.subheader("📄 2. Resume Ingestion (PDF Drag & Drop)")
    uploaded_file = st.file_uploader("Upload Candidate Resume (.pdf)", type=["pdf"], help="Drag and drop a PDF file here")

    extracted_text = ""
    if uploaded_file is not None:
        try:
            pdf_bytes = io.BytesIO(uploaded_file.read())
            pdf_reader = pypdf.PdfReader(pdf_bytes)
            pages_text = [page.extract_text() or "" for page in pdf_reader.pages]
            extracted_text = "\n".join(pages_text).strip()

            if len(extracted_text) < 50:
                st.warning("⚠️ Warning: Extracted text is short. Ensure the PDF contains selectable text.")
            else:
                st.success(f"✅ Ingested `{uploaded_file.name}` ({len(extracted_text):,} characters parsed)")
                with st.expander("🔍 View Raw Parsed Resume Text"):
                    st.text(extracted_text[:800] + "...")
        except Exception as e:
            st.error(f"Failed to read PDF: {e}")
    else:
        st.info("💡 Upload a PDF or the system will use a default fallback resume profile.")
        extracted_text = "5 years Python developer with experience in FastAPI, Docker, PostgreSQL, and basic LangChain."

# ─────────────────────────────────────────────────────────────
# 6. PIPELINE EXECUTION & HUMAN-IN-THE-LOOP CONTROLS
# ─────────────────────────────────────────────────────────────
st.divider()

if st.button("🚀 Run Initial Screening & Assessment", type="primary", use_container_width=True):
    if not api_key:
        st.error("Please provide a valid Groq API Key in the sidebar.")
    elif not extracted_text.strip():
        st.error("Please provide or upload a valid resume.")
    else:
        with st.spinner("Multi-Agent graph running: Screener, Interviewer & Compliance nodes..."):
            try:
                app_graph = build_hr_graph(api_key)
                st.session_state.app_graph = app_graph
                st.session_state.thread_config = {"configurable": {"thread_id": "hr_session_001"}}

                payload = {
                    "job_description": jd_input,
                    "resume_text": extracted_text,
                    "salary_expectation": int(salary_exp),
                    "salary_budget_max": int(budget_max),
                    "work_authorization": work_auth,
                    "screening_result": None,
                    "interview_plan": None,
                    "compliance_result": None,
                    "final_output": None
                }

                # Run until interrupt_before=["hr_ops"]
                for _ in app_graph.stream(payload, config=st.session_state.thread_config):
                    pass

                st.session_state.phase = "hitl_ready"
                st.success("🎯 Initial evaluation completed! Paused at Human-in-the-Loop Review Gate.")

            except Exception as e:
                st.error(f"Execution Error: {e}")

# Display Results & HITL Action
if st.session_state.get("phase") in ["hitl_ready", "completed"]:
    app_graph = st.session_state.app_graph
    config = st.session_state.thread_config
    current_state = app_graph.get_state(config)

    screening = current_state.values.get("screening_result") or {}
    interview = current_state.values.get("interview_plan") or {}
    compliance = current_state.values.get("compliance_result") or {}

    r1, r2, r3 = st.columns(3)
    with r1:
        st.subheader("🔍 Screening Audit")
        st.metric("Match Score", f"{screening.get('match_score', 0)}/100")
        st.write(f"**Decision:** `{screening.get('screening_decision')}`")
        st.write(f"**Matched:** {', '.join(screening.get('matched_skills', []))}")
        st.write(f"**Missing:** {', '.join(screening.get('missing_skills', []))}")

    with r2:
        st.subheader("⚖️ Compliance Audit")
        st.write(f"**Visa Compliant:** {'✅ Yes' if compliance.get('visa_compliant') else '❌ No'}")
        st.write(f"**Budget Fit:** {'✅ Yes' if compliance.get('compensation_band_fit') else '❌ No'}")
        st.info(compliance.get("compliance_notes", "N/A"))

    with r3:
        st.subheader("🎯 Targeted Questions")
        tech_q = interview.get("technical_questions", [])
        for i, q in enumerate(tech_q, 1):
            st.markdown(f"**Q{i}:** {q}")

    # Human-in-the-Loop Action Trigger
    if st.session_state.get("phase") == "hitl_ready":
        st.divider()
        st.markdown("### 🛑 Human-in-the-Loop (HITL) Gate")
        st.warning("Manager review required: Review the agent findings above before generating and dispatching official HR communications.")

        if st.button("✅ Approve & Dispatch HR Ops Agent", type="secondary", use_container_width=True):
            with st.spinner("Resuming execution: HR Ops synthesizing decision memo & email draft..."):
                for _ in app_graph.stream(None, config=config):
                    pass
                st.session_state.phase = "completed"
                st.rerun()

    # Final Output Display
    if st.session_state.get("phase") == "completed":
        st.divider()
        st.subheader("📝 Final HR Ops Action & Drafted Communication")
        final_state = app_graph.get_state(config)
        final_output = final_state.values.get("final_output") or {}

        st.success(f"**FINAL HR DECISION:** `{final_output.get('decision')}`")
        st.text_area("Synthesized Action Item / Email Draft", value=final_output.get("action_item", ""), height=260)

Overwriting app.py


In [ ]:
import subprocess
import time
import urllib.request

# 1. Kill lingering processes on port 8501
!fuser -k 8501/tcp > /dev/null 2>&1
time.sleep(1) # Give some time for processes to terminate

# 2. Get External IP Password
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"🔑 LOCALTUNNEL PASSWORD (IP): {external_ip}")

# 3. Run Streamlit with improved proxy/CORS settings and visible logs
cmd = [
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "true",  # Changed to true for proxy compatibility
    "--server.enableXsrfProtection", "false", # Re-added for robustness
    "--browser.gatherUsageStats", "false"
]
# Removed stdout/stderr redirection to make Streamlit's logs visible
process = subprocess.Popen(cmd)

# Wait for server to bind with increased polling duration
print("⏳ Waiting for Streamlit server to bind to port 8501...")
for _ in range(20): # Increased wait time to 20 seconds
    try:
        response = urllib.request.urlopen("http://127.0.0.1:8501")
        if response.getcode() == 200:
            print("✅ Streamlit server is active and responding!")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Streamlit took longer than expected, starting tunnel anyway...")

# 4. Launch Localtunnel
print("🌐 Public URL below — Click it and enter the IP password above:")
!npx localtunnel --port 8501

🔑 LOCALTUNNEL PASSWORD (IP): 34.12.40.92
⏳ Waiting for Streamlit server to bind to port 8501...
✅ Streamlit server is active and responding!
🌐 Public URL below — Click it and enter the IP password above:
⠙⠹⠸⠼⠴your url is: https://floppy-suns-lead.loca.lt
^C


In [ ]:
# 1. Download Cloudflare tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Kill stale ports
!fuser -k 8501/tcp > /dev/null 2>&1

# 3. Start Streamlit
import subprocess, time
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false"])
time.sleep(3)

# 4. Open instant Cloudflare tunnel (Direct link, no password screens)
!cloudflared tunnel --url http://127.0.0.1:8501

2026-09-01T19:31:11Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-01T19:31:11Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-01T19:31:17Z INF +--------------------------------------------------------------------------------------------+
2026-09-01T19:31:17Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-01T19:31:17Z INF |  https://games-paperbacks-bodies-subscriber.trycloudfl